In [ ]:
import pandas as pd

file_path = "VLibrasBD.csv" 
df = pd.read_csv(file_path, encoding='utf-8')
df.head(10)

In [ ]:
import sys

print("--- STARTING ENVIRONMENT CHECK ---")
print(f"Using Python version: {sys.version}\n")

# --- Checking Main Libraries ---

try:
    import transformers
    print(f"Transformers OK. Installed version: {transformers.__version__}")
except ImportError:
    print("ERROR: 'transformers' library not found.")

try:
    import torch
    print(f"PyTorch (torch) OK. Installed version: {torch.__version__}")
except ImportError:
    print("ERROR: 'torch' library not found.")

try:
    import datasets
    print(f"Datasets OK. Installed version: {datasets.__version__}")
except ImportError:
    print("ERROR: 'datasets' library not found.")

try:
    import accelerate
    print(f"Accelerate OK. Installed version: {accelerate.__version__}")
except ImportError:
    print("ERROR: 'accelerate' library not found.")

try:
    import pandas
    print(f"Pandas OK. Installed version: {pandas.__version__}")
except ImportError:
    print("ERROR: 'pandas' library not found.")

try:
    import sacrebleu
    print(f"Sacrebleu OK. Installed version: {sacrebleu.__version__}")
except ImportError:
    print("ERROR: 'sacrebleu' library not found.")

try:
    import evaluate
    print(f"Evaluate OK. Installed version: {evaluate.__version__}")
except ImportError:
    print("ERROR: 'evaluate' library not found.")


# --- CRITICAL CHECK: GPU Access ---
print("\n--- CHECKING GPU (CUDA) ACCESS ---")
try:
    import torch
    if torch.cuda.is_available():
        print(f"SUCCESS! PyTorch found the GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: PyTorch is installed, but could NOT find your local GPU.")
except Exception as e:
    print(f"ERROR: Failed to check GPU status with PyTorch: {e}")

print("\n--- VERIFICATION COMPLETED ---")

In [ ]:
import os

# The script to be executed
generator_script = "gen_variants.py"

# --- Step 1: Check files ---
print("Contents of the working directory BEFORE execution:")
# !ls -l is the Linux 'ls' command (where you are)
!ls -l

# --- Step 2: Execute the script to generate variants ---
print("\nExecuting the gen_variants.py script...")
!python {generator_script}
print("\nScript execution completed!")

# --- Step 3: Verify generated files ---
print("\nContents of the working directory AFTER execution (there should be a 'variants' folder):")
!ls -l


In [ ]:
# ==============================================================================
# TRAINING SCRIPT - FLAN-T5 BASELINE (VLIBRAS PT -> PGS)
# ==============================================================================

import os
import json
import random
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from bitsandbytes.optim import AdamW8bit
from collections import Counter

from transformers import (
    AutoTokenizer, 
    get_scheduler, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    AutoConfig
)
import datasets                             # Provides dataset handling structures
datasets.config.TORCHVISION_AVAILABLE = False
import sys
if "torchvision" in sys.modules:
    del sys.modules["torchvision"]
import evaluate
from accelerate import Accelerator

# Preventive VRAM cleanup
torch.cuda.empty_cache()

# --- 1. Manual Configuration ---
DATASET_PATH = "./variants"
dataset_variant = "lower"
MSRC_LANG = "pt"
AUX_LANG = "pgs"
RUN = "Optimized_FLAN"  

# Hyperparameters 
BATCH_SIZE = 2                  
gradient_accumulation_steps = 16
N_EPOCHS = 50                                                               
LR_BODY = 5e-5                                                              
LR_EMBEDDINGS = LR_BODY * 10        # ATTENTION!!                                          
LR_SCHEDULER = "linear" 
SRC_LANG = 'pt'   
TGT_LANG = 'pgs'
MAX_INPUT_LENGTH = 256
MAX_TGT_LENGTH = 256

VARIANT = '.' + dataset_variant if dataset_variant != 'plain' else '' 
TRAIN_PATH = os.path.join(DATASET_PATH, f"pt-pgs{VARIANT}.train.csv")
TEST_PATH = os.path.join(DATASET_PATH, f"pt-pgs{VARIANT}.test.csv")
DEV_PATH = os.path.join(DATASET_PATH, f"pt-pgs{VARIANT}.val.csv") 

output_dir = f'runs/{MSRC_LANG}-{AUX_LANG}-{dataset_variant}-run{RUN}'
os.makedirs(output_dir, exist_ok=True)

# Hardware
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
# ==============================================================================
# 2. VOCABULARY EXTRACTION AND FLAN-T5 TOKENIZER
# ==============================================================================
MODEL_NAME = "google/flan-t5-base" # UPDATED TO FLAN-T5

print(" Extracting VLibras gloss vocabulary...")
df_train_vocab = pd.read_csv(TRAIN_PATH)
df_test_vocab = pd.read_csv(TEST_PATH)
all_glosses_text = pd.concat([df_train_vocab['pgs'], df_test_vocab['pgs']]).str.cat(sep=' ')

gloss_counts = Counter(all_glosses_text.split())
MIN_FREQUENCY = 3 
gloss_vocab = [token for token, count in gloss_counts.items() if count >= MIN_FREQUENCY]
print(f"Total glosses: {len(gloss_counts)}. Useful glosses (>=3): {len(gloss_vocab)}")

print(f" Loading Tokenizer: {MODEL_NAME}")
# FLAN-T5 does not use language IDs on initialization
tokenizers = {'pt_X': AutoTokenizer.from_pretrained(MODEL_NAME)}
tokenizers['pt_X'].add_tokens(list(gloss_vocab))

# ==============================================================================
# 3. DATA PREPARATION (WITH T5 TEXTUAL PROMPT)
# ==============================================================================
print(" Loading CSVs...")
raw_datasets = datasets.DatasetDict({
    'train': datasets.Dataset.from_pandas(df_train_vocab),
    'validation': datasets.Dataset.from_pandas(pd.read_csv(DEV_PATH)),
    'test': datasets.Dataset.from_pandas(df_test_vocab)
})

raw_datasets = raw_datasets.filter(lambda ex: pd.notna(ex['pt']) and pd.notna(ex['pgs']) and len(str(ex['pt']).strip()) > 0)

def dynamic_word_dropout_and_tokenize(batch):
    p_drop = 0   # Baseline = 0 dropout                                            
    inputs_dropped = []
    is_training = batch.get('is_train_split', [False])[0] 
    
    # Task guiding prompt
    prefix = "translate Portuguese to LIBRAS Sign Language Glosses: "

    for text in batch["pt"]:
        if is_training and p_drop > 0:
            words = str(text).split()
            kept_words = [w for w in words if random.random() > p_drop]
            if len(kept_words) == 0:
                kept_words = words
            inputs_dropped.append(prefix + " ".join(kept_words))
        else:
            inputs_dropped.append(prefix + str(text))
            
    targets = [str(ex) for ex in batch["pgs"]]
    
    # Unified Tokenization 
    model_inputs = tokenizers['pt_X'](
        inputs_dropped, 
        text_target=targets, 
        max_length=MAX_INPUT_LENGTH, 
        truncation=True
    )
    
    return model_inputs

raw_datasets['train'] = raw_datasets['train'].add_column("is_train_split", [True] * len(raw_datasets['train']))
raw_datasets['validation'] = raw_datasets['validation'].add_column("is_train_split", [False] * len(raw_datasets['validation']))
raw_datasets['test'] = raw_datasets['test'].add_column("is_train_split", [False] * len(raw_datasets['test']))

print(" Configuring Dynamic Tokenization...")
raw_datasets.set_transform(dynamic_word_dropout_and_tokenize)

# ==============================================================================
# 4. BASELINE MODEL & OPTIMIZER
# ==============================================================================
best_bleu = 0.0
checkpoint_config = os.path.join(output_dir, "config.json")

if os.path.exists(checkpoint_config):
    print(f"\n CHECKPOINT FOUND. Resuming from: {output_dir}")
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir, torch_dtype=torch.float16)
    try:
        with open(os.path.join(output_dir, 'stats.json'), 'r') as f:
            best_bleu = max(json.load(f).get('bleu', [0]))
    except: pass
else:
    print(f"\n STARTING FROM SCRATCH with '{MODEL_NAME}'...")
    config = AutoConfig.from_pretrained(MODEL_NAME)
    
    # BASELINE: Zeroed dropout. In FLAN-T5 the parameter is called dropout_rate
    config.dropout_rate = 0.3                                                                        
    config.attention_dropout = 0.2                                        
    
    # We use float16 to ensure VRAM relief
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, config=config, torch_dtype=torch.float16)
    model.resize_token_embeddings(len(tokenizers['pt_X']))
    
    # Additional protection against OOM
    model.gradient_checkpointing_enable()
    model.config.use_cache = False 

# BASELINE: No Differential LR (LR_EMBEDDINGS = LR_BODY)
embedding_params = model.get_input_embeddings().parameters()
embedding_param_ids = {id(p) for p in embedding_params}
body_params = [p for p in model.parameters() if id(p) not in embedding_param_ids]

optimizer_grouped_parameters = [
    {"params": body_params, "lr": LR_BODY},
    {"params": model.get_input_embeddings().parameters(), "lr": LR_EMBEDDINGS},
]
optimizer = AdamW8bit(optimizer_grouped_parameters, weight_decay=0.05)

data_collator = DataCollatorForSeq2Seq(tokenizers['pt_X'], model=model)
train_dataloader = DataLoader(raw_datasets["train"], shuffle=True, collate_fn=data_collator, batch_size=BATCH_SIZE)
eval_dataloader = DataLoader(raw_datasets["validation"], collate_fn=data_collator, batch_size=BATCH_SIZE)

accelerator = Accelerator(gradient_accumulation_steps=gradient_accumulation_steps)
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

num_training_steps = N_EPOCHS * len(train_dataloader) 
# BASELINE: Zeroed Warmup
lr_scheduler = get_scheduler(LR_SCHEDULER, optimizer=optimizer, num_warmup_steps=0.05, num_training_steps=num_training_steps)

def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()
    decoded_preds = tokenizers['pt_X'].batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizers['pt_X'].pad_token_id)
    decoded_labels = tokenizers['pt_X'].batch_decode(labels, skip_special_tokens=True)
    return [p.strip() for p in decoded_preds], [[l.strip()] for l in decoded_labels]

# ==============================================================================
# 5. TRAINING LOOP
# ==============================================================================
patience = 10                                                                                                                              
patience_counter = 0
progress_bar = tqdm(range(num_training_steps))
writer = SummaryWriter(log_dir=f'{output_dir}/logs')
stats = {'bleu': []}

for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0
    # BASELINE: No Label Smoothing
    loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100) 
    
    for batch in train_dataloader:
        outputs = model(**batch)

        
        logits = outputs.logits
        labels = batch["labels"]
        
        loss = loss_fct(logits.transpose(1, 2), labels)
        
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)
        
        if (progress_bar.n + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            
        total_loss += loss.item() * gradient_accumulation_steps
        progress_bar.update(1)
    
    avg_loss = total_loss / len(train_dataloader)
    writer.add_scalar('Training/Loss', avg_loss, epoch)

    # --- VALIDATION ---import datasets
    model.eval()
    metric = evaluate.load("sacrebleu") 
    
    for batch in tqdm(eval_dataloader, desc=f"Validation Epoch {epoch+1}", leave=False):
        with torch.no_grad():
            # FLAN-T5: Does not use forced_bos_token_id
            # BASELINE: num_beams = 1 (Greedy Search)
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"], 
                attention_mask=batch["attention_mask"], 
                max_length=MAX_TGT_LENGTH,
                num_beams=5 
            )
        
        labels = batch["labels"]
        generated_tokens = accelerator.pad_across_processes(generated_tokens, dim=1, pad_index=tokenizers['pt_X'].pad_token_id)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)
        
        preds_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)
        decoded_preds, decoded_labels = postprocess(preds_gathered, labels_gathered)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    results = metric.compute(tokenize='none', lowercase=True)
    current_bleu = results['score']
    
    b1, b2, b3, b4 = results['precisions']
    print(f"\n--- Epoch {epoch+1} ---")
    print(f" Loss: {avg_loss:.4f} |  SacreBLEU: {current_bleu:.2f} (Best: {best_bleu:.2f})")
    print(f"   Isolated BLEUs -> BLEU-1: {b1:.2f} | BLEU-2: {b2:.2f} | BLEU-3: {b3:.2f} | BLEU-4: {b4:.2f}")
    
    writer.add_scalar('BLEU/Validation', current_bleu, epoch)
    stats['bleu'].append(current_bleu)

    if current_bleu > best_bleu:
        print(f" New Record! Saving model...")
        best_bleu = current_bleu
        patience_counter = 0
        accelerator.wait_for_everyone()
        accelerator.unwrap_model(model).save_pretrained(output_dir, save_function=accelerator.save)
        tokenizers['pt_X'].save_pretrained(output_dir)
        with open(os.path.join(output_dir, 'stats.json'), 'w') as f: json.dump(stats, f)
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print(" Early stopping.")
        break

writer.close()

In [ ]:
# ==============================================================================
# INFERENCE AND EVALUATION SCRIPT - FLAN-T5 BASELINE (VLIBRASBD)
# BLEU (1 to 4), SACREBLEU AND ROUGE-L CALCULATION
# ==============================================================================
import os
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Mandatory imports for standardized metrics
import sacrebleu
from sacrebleu.metrics import BLEU
import evaluate

# Clears VRAM residue from previous trainings
torch.cuda.empty_cache()

# --- 1. PATH AND VARIABLE CONFIGURATION ---
DATASET_PATH = "./variants"
dataset_variant = "lower"
MSRC_LANG = "pt"
AUX_LANG = "pgs"

# POINTS TO THE CORRECT VLIBRAS PHASE FOLDER
RUN = "Optimized_FLAN" 

SRC_LANG = 'pt'
TGT_LANG = 'pgs'
VARIANT = '.' + dataset_variant if dataset_variant != 'plain' else '' 
TEST_PATH = os.path.join(DATASET_PATH, f"pt-pgs{VARIANT}.val.csv")
output_dir = f'runs/{MSRC_LANG}-{AUX_LANG}-{dataset_variant}-run{RUN}'

# --- 2. INFERENCE PARAMETERS ---
MAX_INPUT_LENGTH = 256
MAX_TGT_LENGTH = 256
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"\n STARTING FLAN-T5 INFERENCE (VLIBRASBD)")
print(f" Looking for model in: {output_dir}")
print(f" Reading test data from: {TEST_PATH}")

# --- 3. MODEL AND TOKENIZER LOADING ---
if not os.path.exists(output_dir):
    raise FileNotFoundError(f" Model folder not found: {output_dir}. Check the RUN name.")

# CRITICAL CORRECTION: Loads the model in float16 to avoid OOM
best_model = AutoModelForSeq2SeqLM.from_pretrained(output_dir, torch_dtype=torch.float16).to(DEVICE)
best_tokenizer = AutoTokenizer.from_pretrained(output_dir)

best_model.eval()

# --- 4. DATA READING AND PREPARATION ---
df_test_final = pd.read_csv(TEST_PATH)
df_test_final = df_test_final.dropna(subset=[SRC_LANG, TGT_LANG])

preds_final = []
refs_final = []

# --- 5. GENERATION LOOP ---
# THE FLAN-T5 SECRET: Task guiding prompt
prefix = "translate Portuguese to LIBRAS Sign Language Glosses: "

for index, row in tqdm(df_test_final.iterrows(), total=len(df_test_final), desc="Generating Translations"):
    # Concatenates the textual prompt to the input sentence
    src_text = prefix + str(row[SRC_LANG])
    tgt_text = str(row[TGT_LANG])
    
    inputs = best_tokenizer(src_text, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True).to(DEVICE)
    
    with torch.no_grad():
        outputs = best_model.generate(
            **inputs, 
            max_length=MAX_TGT_LENGTH,
            # FLAN-T5 does not use forced_bos_token_id
            num_beams=5,                  # MANDATORY BASELINE: Greedy Search
            early_stopping=True           
        )
        
    pred_str = best_tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    preds_final.append(pred_str)
    refs_final.append(tgt_text) 

# --- 6. METRICS CALCULATION ---
print("\n Calculating purist metrics...")

# Formats required by each library
refs_transposed = [refs_final]  
refs_flat = refs_final          

results = {}

# A) SacreBLEU (Canonical) - TOKENIZE NONE MANDATORY FOR GLOSSES
sb = sacrebleu.corpus_bleu(preds_final, refs_transposed, lowercase=True, tokenize='none')
results['SacreBLEU'] = sb.score

# B) Granular BLEU 1, 2, 3, 4
results['BLEU-1'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=1).corpus_score(preds_final, refs_transposed).score
results['BLEU-2'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=2).corpus_score(preds_final, refs_transposed).score
results['BLEU-3'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=3).corpus_score(preds_final, refs_transposed).score
results['BLEU-4'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=4).corpus_score(preds_final, refs_transposed).score

# C) ROUGE-L (Recall-Oriented)
try:
    rouge = evaluate.load("rouge")
    rouge_res = rouge.compute(predictions=preds_final, references=refs_flat)
    results['ROUGE-L'] = rouge_res['rougeL'] * 100 
except Exception as e:
    print(f" Warning: Could not calculate ROUGE ({e})")
    results['ROUGE-L'] = 0.0

# --- 7. RESULTS DISPLAY ---
print("\n" + "="*40)
print(f" FINAL PERFORMANCE REPORT (FLAN-T5 BASELINE)")
print("="*40)
print(f"{'Metric':<15} | {'Score':<10}")
print("-" * 28)
print(f"{'BLEU-1 ':<15} | {results['BLEU-1']:.2f}")
print(f"{'BLEU-2 ':<15} | {results['BLEU-2']:.2f}")
print(f"{'BLEU-3 ':<15} | {results['BLEU-3']:.2f}")
print(f"{'BLEU-4 ':<15} | {results['BLEU-4']:.2f}")
print("-" * 28)
print(f"{'SacreBLEU':<15} | {results['SacreBLEU']:.2f}")
print(f"{'ROUGE-L':<15} | {results['ROUGE-L']:.2f}")
print("="*40)

# --- 8. DETAILED SAVING ---
df_result_file = pd.DataFrame({
    'pt': df_test_final['pt'], 
    'pgs_ref': refs_final, 
    'pgs_pred': preds_final
})

csv_out = os.path.join(output_dir, 'final_metrics_evaluation.csv')
df_result_file.to_csv(csv_out, index=False)
print(f"\nDetailed file with translations saved at: {csv_out}")

In [ ]:
##########################################################################################################################################################